In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

## Config

In [2]:
import json
from pathlib import Path

DEVICE = "cuda"
MODEL_ID = "stabilityai/stable-diffusion-3-medium-diffusers"
NUM_STEPS = 28
GUIDANCE_SCALE = 7.0

N_STEPS = 28
N_BLOCKS = 24
COMPONENTS = ["L_minus_1", "delta_attn", "delta_mlp"]

ATTR_SEEDS = list(range(5))       # seeds per attribute prompt
CONCEPT_SEEDS = list(range(30))   # seeds for bare-concept prompt

PROMPT_DIR = Path("../prompt")
OUTPUT_DIR = Path("./output/decomposed_multi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JSON_FILES = sorted(PROMPT_DIR.glob("*.json"))
print(f"found {len(JSON_FILES)} concept JSON files:")
for p in JSON_FILES:
    d = json.load(open(p))
    n_attr = sum(len(v) for v in d["simple_attribute_prompts"].values())
    print(f"  - {p.name}: concept=\"{d['concept']}\", {n_attr} attrs")

found 8 concept JSON files:
  - airplane.json: concept="airplane", 16 attrs
  - bird.json: concept="bird", 15 attrs
  - car.json: concept="car", 13 attrs
  - clock.json: concept="clock", 14 attrs
  - couch.json: concept="couch", 23 attrs
  - elephant.json: concept="elephant", 13 attrs
  - train.json: concept="train", 13 attrs
  - umbrella.json: concept="umbrella", 16 attrs


## Load Pipeline

In [3]:
import torch
from diffusers import StableDiffusion3Pipeline

pipe = StableDiffusion3Pipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

/home/haeun/miniconda3/envs/C3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading pipeline components...:  22%|██▏       | 2/9 [00:00<00:02,  2.46it/s]

Loading pipeline components...:  56%|█████▌    | 5/9 [00:01<00:00,  5.73it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


...

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.41it/s]

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.32it/s]


Loading pipeline components...:  78%|███████▊  | 7/9 [00:02<00:00,  2.42it/s]

Loading pipeline components...: 100%|██████████| 9/9 [00:02<00:00,  3.78it/s]

Loading pipeline components...: 100%|██████████| 9/9 [00:02<00:00,  3.64it/s]

## Hook & Capture

L = L_minus_1 + delta_attn + delta_mlp (cond branch = index 1). Disk 저장 X — run 끝에 CPU fp16 으로 옮긴 뒤 바로 running-sum 에 누적.

In [4]:
import gc
from functools import wraps
from collections import defaultdict

# ── 결과 버퍼 (한 run 동안만 채워짐) ──
captured = defaultdict(dict)  # captured[step][block] = {"L_minus_1":..., "delta_attn":..., "delta_mlp":...}
_active = {"on": False}
denoising_step = [0]

# ── transformer.forward wrap (step 추적) ──
if not getattr(pipe.transformer.forward, "_is_step_tracked", False):
    _orig_transformer_forward = pipe.transformer.forward

    @wraps(_orig_transformer_forward)
    def _step_tracked_forward(*args, **kwargs):
        result = _orig_transformer_forward(*args, **kwargs)
        denoising_step[0] += 1
        return result

    _step_tracked_forward._is_step_tracked = True
    pipe.transformer.forward = _step_tracked_forward
    print("step tracker attached.")
else:
    print("step tracker already attached.")

# ── 블록 patch ──
already_patched = 0
for block_idx, block in enumerate(pipe.transformer.transformer_blocks):
    if getattr(block.forward, "_is_decomposed_patched", False):
        already_patched += 1
        continue

    original_forward = block.forward

    def _make_patched(orig_fwd, idx, blk):
        @wraps(orig_fwd)
        def patched_forward(hidden_states, encoder_hidden_states, temb, *args, **kwargs):
            if not _active["on"]:
                return orig_fwd(hidden_states, encoder_hidden_states, temb, *args, **kwargs)

            s = denoising_step[0]
            L_minus_1 = hidden_states.detach().clone()

            norm2_buf = {}
            def _capture_norm2(module, args_in):
                if "val" not in norm2_buf and args_in:
                    norm2_buf["val"] = args_in[0].detach().clone()

            handle = blk.norm2.register_forward_pre_hook(_capture_norm2)
            try:
                block_output = orig_fwd(hidden_states, encoder_hidden_states, temb, *args, **kwargs)
            finally:
                handle.remove()

            L_output = block_output[1].detach() if isinstance(block_output, tuple) else block_output.detach()

            if "val" in norm2_buf:
                L_m1  = L_minus_1[1].cpu().float()
                L_aa  = norm2_buf["val"][1].cpu().float()
                L_out = L_output[1].cpu().float()
                captured[s][idx] = {
                    "L_minus_1":  L_m1.half(),
                    "delta_attn": (L_aa - L_m1).half(),
                    "delta_mlp":  (L_out - L_aa).half(),
                }
            return block_output

        patched_forward._is_decomposed_patched = True
        return patched_forward

    block.forward = _make_patched(original_forward, block_idx, block)

if already_patched > 0:
    print(f"⚠ {already_patched} blocks already patched — restart kernel for re-patch")
else:
    print(f"{len(pipe.transformer.transformer_blocks)} blocks patched.")
print("hook ready.")

step tracker attached.
24 blocks patched.
hook ready.


## Run helpers

In [5]:
def run_once(prompt_text, seed):
    """single inference → returns (captured_cpu, image)
    captured_cpu: list[step][block] = {"L_minus_1": fp16 CPU, "delta_attn": fp16 CPU, "delta_mlp": fp16 CPU}
    GPU capture buffer cleared before return."""
    captured.clear()
    denoising_step[0] = 0
    _active["on"] = True

    generator = torch.Generator(device=pipe.device).manual_seed(seed)
    image = pipe(
        prompt_text,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    ).images[0]
    _active["on"] = False

    # captured 는 이미 CPU fp16 (hook 내부에서 .cpu() 처리됨)
    out = [[None]*N_BLOCKS for _ in range(N_STEPS)]
    for s in range(N_STEPS):
        for b in range(N_BLOCKS):
            d = captured[s][b]
            out[s][b] = {k: v.flatten() for k, v in d.items()}

    captured.clear()
    torch.cuda.empty_cache()
    gc.collect()
    return out, image


def make_sums():
    """empty running-sum container: sums[comp][step][block] = None (채워지면 fp32 CPU 1D)"""
    return {c: [[None]*N_BLOCKS for _ in range(N_STEPS)] for c in COMPONENTS}


def accumulate_sum(sums, captured_cpu):
    """sums[comp][s][b] (fp32 CPU) += captured_cpu[s][b][comp] (fp16 CPU)"""
    for s in range(N_STEPS):
        for b in range(N_BLOCKS):
            for c in COMPONENTS:
                v = captured_cpu[s][b][c].float()
                if sums[c][s][b] is None:
                    sums[c][s][b] = v
                else:
                    sums[c][s][b] += v


def cos_sim_heatmap(sum_A, sum_B):
    """cos(mean_A, mean_B) = cos(sum_A, sum_B) per component. → dict[comp] -> (28,24) ndarray"""
    import numpy as np
    out = {}
    for c in COMPONENTS:
        h = np.zeros((N_STEPS, N_BLOCKS), dtype=np.float32)
        for s in range(N_STEPS):
            for b in range(N_BLOCKS):
                a = sum_A[c][s][b]
                cc = sum_B[c][s][b]
                dot = (a * cc).sum().item()
                na = a.norm().item()
                nc = cc.norm().item()
                h[s, b] = dot / max(na * nc, 1e-12)
        out[c] = h
    return out

## Plot helpers (grid PNG 저장 전용)

In [6]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


def save_concept_grid(imgs, concept, out_path, cols=6):
    n = len(imgs)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.5))
    axes_flat = axes.flat if rows*cols > 1 else [axes]
    for i, ax in enumerate(axes_flat):
        if i < n:
            ax.imshow(imgs[i]); ax.set_title(f"seed {i}", fontsize=8)
        ax.axis("off")
    fig.suptitle(f'"{concept}" — {n} seeds', fontsize=14)
    plt.tight_layout()
    plt.savefig(out_path, dpi=110)
    plt.close(fig)


def save_attr_grid(attr_imgs_dict, prompts_dict, concept, out_path, cols=5):
    """attr_imgs_dict: {(attr_type, attr_value): [PIL, PIL, ...]}
       prompts_dict:   {(attr_type, attr_value): prompt_text}"""
    rows = len(attr_imgs_dict)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.2, rows*2.4), squeeze=False)
    for r, ((at, av), imgs) in enumerate(attr_imgs_dict.items()):
        for c in range(cols):
            ax = axes[r, c]
            ax.axis("off")
            if c < len(imgs):
                ax.imshow(imgs[c])
                if c == 0:
                    ax.set_ylabel(f"{at}/{av}", fontsize=8, rotation=0, ha="right", va="center")
                    ax.set_title(prompts_dict[(at, av)], fontsize=7, loc="left")
                else:
                    ax.set_title(f"seed {c}", fontsize=7)
    fig.suptitle(f'"{concept}" — attribute prompts × {cols} seeds', fontsize=13)
    plt.tight_layout()
    plt.savefig(out_path, dpi=110)
    plt.close(fig)


def save_cossim_heatmap(cos_maps_per_attr, concept, out_path):
    """cos_maps_per_attr: {(attr_type, attr_value): {comp: (28,24) ndarray}}
       rows = attrs, cols = components. 컬러스케일은 컴포넌트 단위로 통일."""
    attrs = list(cos_maps_per_attr.keys())
    rows = len(attrs); cols = len(COMPONENTS)

    comp_vrange = {}
    for comp in COMPONENTS:
        vals = np.concatenate([cos_maps_per_attr[a][comp].ravel() for a in attrs])
        comp_vrange[comp] = (vals.min(), vals.max())

    fig, axes = plt.subplots(rows, cols, figsize=(cols*4.5, rows*3.6), squeeze=False)
    for r, (at, av) in enumerate(attrs):
        for c, comp in enumerate(COMPONENTS):
            ax = axes[r, c]
            vmin, vmax = comp_vrange[comp]
            im = ax.imshow(cos_maps_per_attr[(at, av)][comp], aspect="auto",
                           cmap="RdYlBu_r", origin="lower", vmin=vmin, vmax=vmax)
            ax.set_xlabel("block"); ax.set_ylabel("step")
            ax.set_title(f"{comp} | {at}/{av}", fontsize=9)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f'Cosine Sim — mean("{concept}" x{len(CONCEPT_SEEDS)}) vs mean(attr x{len(ATTR_SEEDS)})',
                 fontsize=13)
    plt.tight_layout()
    plt.savefig(out_path, dpi=110)
    plt.close(fig)

## Per-JSON experiment

한 JSON 당:
1. concept prompt × 30 seeds 돌리며 `concept_sums` (3 components, CPU fp32) 누적 + 이미지 모아두기
2. 각 attribute 에 대해 × 5 seeds 돌리며 `attr_sums` 누적 + 이미지 모아두기 → 3-component cos-sim heatmap 계산 → attr_sums 해제
3. 세 가지 PNG 저장: `{concept}_concept_grid.png`, `{concept}_attr_grid.png`, `{concept}_cossim_heatmap.png`
4. concept_sums / images 해제

피크 CPU mem ≈ 2 × 3 components × (28·24·4096·1536 × 4 bytes) ≈ 100 GB / JSON.

In [7]:
import time

def run_experiment_for_json(json_path):
    data = json.load(open(json_path))
    concept = data["concept"]
    attr_items = []
    for at, d in data["simple_attribute_prompts"].items():
        for av, pt in d.items():
            attr_items.append((at, av, pt))

    print(f"\n=== {json_path.name} (concept={concept}, {len(attr_items)} attrs) ===")
    t0 = time.time()

    # ── 1) concept run ──
    print(f"  [concept] {len(CONCEPT_SEEDS)} seeds...")
    concept_sums = make_sums()
    concept_imgs = []
    for i, seed in enumerate(CONCEPT_SEEDS):
        captured_cpu, image = run_once(concept, seed)
        accumulate_sum(concept_sums, captured_cpu)
        concept_imgs.append(image)
        del captured_cpu
        if (i+1) % 10 == 0 or i+1 == len(CONCEPT_SEEDS):
            print(f"    {i+1}/{len(CONCEPT_SEEDS)}  ({time.time()-t0:.1f}s)")

    # concept grid PNG 저장 + 이미지 메모리 해제
    save_concept_grid(concept_imgs, concept, OUTPUT_DIR / f"{concept}_concept_grid.png")
    del concept_imgs
    gc.collect()

    # ── 2) attribute runs + cos sim ──
    cos_maps_per_attr = {}           # {(at, av): {comp: (28,24)}}
    attr_imgs_dict = {}              # {(at, av): [PIL x 5]}
    prompts_dict = {}
    for ai, (at, av, ptxt) in enumerate(attr_items):
        attr_sums = make_sums()
        imgs = []
        for seed in ATTR_SEEDS:
            captured_cpu, image = run_once(ptxt, seed)
            accumulate_sum(attr_sums, captured_cpu)
            imgs.append(image)
            del captured_cpu

        cos_maps_per_attr[(at, av)] = cos_sim_heatmap(attr_sums, concept_sums)
        attr_imgs_dict[(at, av)] = imgs
        prompts_dict[(at, av)] = ptxt

        del attr_sums
        gc.collect()
        comps_range = {c: (cos_maps_per_attr[(at,av)][c].min(), cos_maps_per_attr[(at,av)][c].max()) for c in COMPONENTS}
        print(f"  [{ai+1}/{len(attr_items)}] {at}/{av}  ({time.time()-t0:.1f}s)")

    del concept_sums
    gc.collect()

    # ── 3) attr grid + cos sim heatmap PNG 저장 ──
    save_attr_grid(attr_imgs_dict, prompts_dict, concept, OUTPUT_DIR / f"{concept}_attr_grid.png")
    save_cossim_heatmap(cos_maps_per_attr, concept, OUTPUT_DIR / f"{concept}_cossim_heatmap.png")

    del attr_imgs_dict, cos_maps_per_attr
    gc.collect()

    print(f"  saved: {concept}_concept_grid.png, {concept}_attr_grid.png, {concept}_cossim_heatmap.png")
    print(f"  total {time.time()-t0:.1f}s")

## Run across all JSON files

In [8]:
for jp in JSON_FILES:
    run_experiment_for_json(jp)

print(f"\n==> all done. outputs in {OUTPUT_DIR}")


=== airplane.json (concept=airplane, 16 attrs) ===
  [concept] 30 seeds...


    10/30  (2404.2s)


    20/30  (4681.5s)


    30/30  (6804.5s)


  [1/16] color/blue  (7926.2s)


...

  [13/16] pattern/striped  (17582.9s)


  [14/16] pattern/polka-dotted  (18746.5s)


  [15/16] pattern/floral  (19890.1s)


  [16/16] pattern/geometric  (21048.2s)


  saved: umbrella_concept_grid.png, umbrella_attr_grid.png, umbrella_cossim_heatmap.png
  total 21070.1s

==> all done. outputs in output/decomposed_multi
